# **Семинар 6.** Механизм внимания. Трансформер

In [ ]:
import pandas as pd

train = pd.read_csv("data/train.tsv", sep="\t")
train.head(5)

In [ ]:
test = pd.read_csv("data/test.tsv", sep="\t")

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors, trainers

ru_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
ru_tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
ru_tokenizer.decoder = decoders.Metaspace()

ru_trainer = trainers.BpeTrainer(
    vocab_size=5_000,
    show_progress=False,
    special_tokens=["<pad>", "<unk>"],
)
ru_tokenizer.train_from_iterator(train["ru"], trainer=ru_trainer)

In [ ]:
ru_text_example = train["ru"].iloc[221]
print(ru_text_example)

ru_tokenizer.encode(ru_text_example).tokens

In [ ]:
en_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
en_tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
en_tokenizer.decoder = decoders.Metaspace()

en_trainer = trainers.BpeTrainer(
    vocab_size=5_000,
    show_progress=False,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
)
en_tokenizer.train_from_iterator(train["en"], trainer=en_trainer)

In [ ]:
en_tokenizer.post_processor = processors.TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", en_tokenizer.token_to_id("<bos>")),
        ("<eos>", en_tokenizer.token_to_id("<eos>")),
    ],
)

In [ ]:
en_text_example = train["en"].iloc[221]
print(en_text_example)

en_tokenizer.encode(en_text_example).tokens

In [ ]:
import torch
from torch.utils.data import Dataset


class TranslationDataset(Dataset):
    def __init__(
        self,
        data: pd.DataFrame,
        source_tokenizer: Tokenizer,
        target_tokenizer: Tokenizer,
        source_key: str = "source",
        target_key: str = "target",
    ) -> None:
        super().__init__()

        self.source_key = source_key
        self.target_key = target_key

        self.source_tokenizer = source_tokenizer
        self.target_tokenizer = target_tokenizer

        self.source_ids = [source_tokenizer.encode(text).ids for text in data[source_key]]
        self.target_ids = [target_tokenizer.encode(text).ids for text in data[target_key]]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return torch.tensor(self.source_ids[idx]), torch.tensor(self.target_ids[idx])

    def __len__(self) -> int:
        return len(self.source_ids)


train_dataset = TranslationDataset(train, ru_tokenizer, en_tokenizer, source_key="ru", target_key="en")
train_dataset[42]

In [ ]:
test_dataset = TranslationDataset(test, ru_tokenizer, en_tokenizer, source_key="ru", target_key="en")

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader


def collate(
    batch: list[tuple[torch.Tensor, torch.Tensor]],
    source_pad_token_id: int = 0,
    target_pad_token_id: int = 0,
) -> tuple[torch.Tensor, torch.Tensor]:
    source_ids, target_ids = zip(*batch, strict=True)

    source_ids = pad_sequence(source_ids, batch_first=True, padding_value=source_pad_token_id)
    target_ids = pad_sequence(target_ids, batch_first=True, padding_value=target_pad_token_id)

    return source_ids, target_ids


train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate)
batch = next(iter(train_dataloader))

In [ ]:
batch[0]

In [ ]:
batch[1]

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=128, collate_fn=collate)

## Attention Mechanism

Архитектура трансформер (англ. *transformer*) впервые была представлена в статье [Attention is All You Need](https://arxiv.org/abs/1706.03762), опубликованной в 2017 году. Как следует из названия, ключевым элементом архитектуры трансформер является внимание (англ. *attention mechanism*), поэтому обратим на него особое внимание.

Механизм attention в трансформере принимает на вход три основных компонента: запросы $Q$ (англ. *queries*), ключи $K$ (англ. *keys*) и значения $V$ (англ. *values*). Каждый ключ связан с некоторым значением. Для данного запроса вычисляется его скалярное произведение со всеми ключами (чем сильнее запрос похож на ключ, тем больше скалярное произведение). К полученным числам далее применяется softmax, результат называется весами внимания (англ. *attention weights*). Наконец, новое представление формируется как сумма значений, умноженных на соответствующие коэффициенты:

$$
\operatorname{Attention}(Q, K, V)
=
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V,
$$

где $d_k$ — размерность запросов и ключей.


In [ ]:
def attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Args:
        q (torch.Tensor): queries of shape (B, Lq, Dk)
        k (torch.Tensor): keys of shape (B, Lk, Dk)
        v (torch.Tensor): values of shape (B, Lk, Dv)

    Returns:
        torch.Tensor: attention output of shape (B, Lq, Dv)
    """
    assert q.shape[-1] == k.shape[-1]

    scale_factor = 1 / q.shape[-1] ** 0.5

    attention_weights = q @ k.transpose(-2, -1) * scale_factor  # (B, Lq, Lk)
    attention_weights = torch.softmax(attention_weights, dim=-1)

    return attention_weights @ v

In [ ]:
q = torch.tensor(
    [
        [
            [1.0, 0.0],
            [0.0, 1.0],
        ]
    ]
)  # (1, 2, 2)
k = torch.tensor(
    [
        [
            [1.0, 0.0],
            [0.0, 1.0],
            [1.0, 1.0],
        ]
    ]
)  # (1, 3, 2)
v = torch.tensor(
    [
        [
            [1.0, 0.0, 0.0],
            [0.0, 2.0, 0.0],
            [0.0, 0.0, 3.0],
        ]
    ]
)  # (1, 3, 3)

attention(q, k, v)

Сравним с эталонной реализацией — `torch.nn.functional.scaled_dot_product_attention`:

In [ ]:
from torch import nn

nn.functional.scaled_dot_product_attention(q, k, v)

На практике используют многоголовое внимание (англ. *multi-head attention*, MHA): после линейных проекций векторы запросов, ключей и значений делят на несколько голов и вычисляют attention независимо внутри каждой из них. Затем результаты голов объединяют и применяют выходную линейную проекцию. В PyTorch этот механизм реализован в классе `torch.nn.MultiheadAttention`.

В encoder-decoder трансформере используются три вида attention: encoder self-attention, decoder self-attention и cross-attention. Разберем их по порядку.

Для начала отметим, что в self-attention $Q$, $K$ и $V$ получаются из одних и тех же векторов при помощи разных обучаемых линейных проекций. Таким образом, последовательность обращает внимание сама на себя, поэтому механизм и называется self-attention.

### Encoder Self-Attention

Этот механизм позволяет токенам входной последовательности «посмотреть» друг на друга: $Q$, $K$ и $V$ получаются через линейные проекции векторов, поступающих в текущий слой энкодера.

Заметим, что некоторые позиции во входной последовательности могут быть заняты padding-токенами. Поскольку это технические токены, остальные токены не должны обращать на них внимание. Для этого нам потребуется специальная padding mask.

### Decoder Self-Attention

Здесь $Q$, $K$ и $V$ также получаются через линейные проекции одних и тех же векторов, но уже из последовательности, поступающей в декодер.

Декодер будет генерировать перевод шаг за шагом, используя только уже известные токены. Поэтому во время обучения токены не должны «смотреть» в будущее, даже если весь правильный перевод уже подан на вход. Для этого используется специальная causal mask. При наличии padding-токенов нам также потребуется padding mask.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int, dropout_p: float, is_causal: bool = False) -> None:
        super().__init__()

        if embedding_dim % num_heads != 0:
            msg = "`embedding_dim` must be divisible by `num_heads`."
            raise ValueError(msg)

        self.num_heads = num_heads
        self.dropout_p = dropout_p
        self.is_causal = is_causal

        self.mha = nn.MultiheadAttention(embedding_dim, num_heads, dropout_p, batch_first=True)

    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor | None = None) -> torch.Tensor:
        attn_mask = None

        if self.is_causal:
            length = x.shape[1]

            # causal mask
            attn_mask = torch.ones(length, length, dtype=torch.bool, device=x.device).triu(diagonal=1)
            # attn_mask = x.new_full((length, length), -torch.inf).triu(diagonal=1)

        return self.mha(
            x, x, x,
            key_padding_mask=padding_mask,
            need_weights=False,
            attn_mask=attn_mask,
            is_causal=self.is_causal,
        )[0]  # fmt: skip

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

length = 8
causal_mask = torch.ones(length, length, dtype=torch.bool).triu(diagonal=1)

plt.figure(figsize=(4, 4))
sns.heatmap(
    causal_mask.numpy(),
    annot=True,
    annot_kws={"fontsize": 8},
    fmt="",
    cmap="gray_r",
    vmin=0,
    vmax=1,
    cbar=False,
    square=True,
    linewidths=0.5,
    linecolor="gray",
)
plt.xlabel("K position")
plt.ylabel("Q position")

plt.title("Causal mask (True = attention is not allowed)", fontdict={"fontsize": 10})

plt.tight_layout()
plt.show()

### Cross-Attention

Этот механизм позволяет декодеру обращаться к исходной последовательности. Запросы $Q$ получаются через линейную проекцию векторов декодера после self-attention, а ключи $K$ и значения $V$ — через линейные проекции выхода последнего слоя энкодера.

Таким образом, при генерации перевода декодер может обращать внимание на нужные части исходного текста. Здесь также потребуется padding mask, чтобы игнорировать padding-позиции исходной последовательности.

## Feed-Forward Network

Вот вам еще один синоним для термина multilayer perceptron, не благодарите... Feed-Forward Network (FFN) применяется к вектору каждого токена независимо, с одними и теми же весами.

В классической архитектуре трансформер FFN содержит **два линейных слоя**, разделенных нелинейной функцией активации. Внутренняя размерность при этом обычно кратно превышает размерность входного вектора (например, в 4 раза).

В современных LLM широкое распространение получили gated-активации, например [SwiGLU](https://arxiv.org/pdf/2002.05202).

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, embedding_dim: int, inner_dim: int, dropout_p: float = 0.0) -> None:
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, inner_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(inner_dim, embedding_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


x = torch.randn(1, 8, 128)

ffn = FeedForwardNetwork(128, 512, dropout_p=0.2)
ffn(x).shape

## Layer Normalization

Для стабилизации обучения в трансформерах используется нормализация слоя (англ. layer normalization). Вектор на каждой позиции нормализуется независимо: из его компонент вычитается их среднее, а результат делится на стандартное отклонение. Затем применяются обучаемые масштабирование и сдвиг:

$$
y = \frac{x - \mathrm{E}[x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} \odot \gamma + \beta,
$$

где $\mathrm{E}[x]$ и $\mathrm{Var}[x]$ — среднее и дисперсия компонент вектора $x$, $\gamma$ и $\beta$ — обучаемые векторы того же размера, что и $x$, а $\epsilon$ — небольшая положительная константа для численной устойчивости.

In [ ]:
x = torch.tensor(
    [
        [1.0, 1.0, 5.0, 5.0],
        [1.0, 3.0, 3.0, 9.0],
    ]
)

print(f"{x.mean(dim=-1) = }")
print(f"{x.std(dim=-1, correction=0) = }")

nn.functional.layer_norm(
    x,
    normalized_shape=(4,),
    # weight=torch.tensor([0.0, 1.0, 1.0, 0.0]),
    # bias=torch.tensor([0.1, 0.2, 0.3, 0.4]),
)

Кроме функционального API, в PyTorch также есть класс `torch.nn.LayerNorm` для использования в нейронных сетях. Напишем его упрощенный аналог.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, embedding_dim: int, eps: float = 1e-5) -> None:
        super().__init__()

        self.weight = nn.Parameter(torch.ones(embedding_dim))
        self.bias = nn.Parameter(torch.zeros(embedding_dim))

        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var, mean = torch.var_mean(x, dim=-1, correction=0, keepdim=True)

        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias


layer_norm = LayerNorm(4)
layer_norm(x)

**Примечание:** в современных LLM часто используется другой вариант нормализации — [RMSNorm](https://docs.pytorch.org/docs/main/generated/torch.nn.RMSNorm.html).

## Transformer

Теперь мы готовы собрать один слой (или блок) трансформера. Он будет состоять из multi-head self-attention и feed-forward network, дополненных остаточными связями (англ. *residual connections*) и нормализацией. Накладывая множество таких слоев друг на друга, мы можем увеличивать глубину и размер модели. Например, в Llama 3 (8B) используется 32 слоя, а в GPT-3 (175B) — 96.

**Примечание:** в оригинальной статье *Attention Is All You Need* авторы применяли layer normalization после сложения входа и выхода каждого подслоя — такой подход называется post-layer normalization (post-LN). Однако позднее в работе [On Layer Normalization in the Transformer Architecture](https://proceedings.mlr.press/v119/xiong20b.html) было показано, что перенос нормализации на вход attention и feed-forward network делает обучение более устойчивым. Такой подход называется pre-layer normalization (pre-LN). Этот вариант широко используется в больших языковых моделях, поэтому мы реализуем именно его.

<img src="images/transformer-01-pre-post-ln.png" alt="Post-LN (слева) и Pre-LN (справа)" width="400">


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        embedding_dim: int,
        inner_dim: int,
        num_heads: int,
        is_causal: bool = False,
        dropout_p: float = 0.0,
    ) -> None:
        super().__init__()

        self.pre_attn_norm = LayerNorm(embedding_dim)
        self.attn = SelfAttention(
            embedding_dim,
            num_heads,
            dropout_p=dropout_p,
            is_causal=is_causal,
        )

        self.pre_ffn_norm = LayerNorm(embedding_dim)
        self.ffn = FeedForwardNetwork(embedding_dim, inner_dim, dropout_p)

    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.attn(self.pre_attn_norm(x), padding_mask)
        x = x + self.ffn(self.pre_ffn_norm(x))

        return x

In [ ]:
class TransformerDecoderBlock(TransformerBlock):
    def __init__(
        self,
        embedding_dim: int,
        inner_dim: int,
        num_heads: int,
        dropout_p: float = 0.0,
    ) -> None:
        super().__init__(
            embedding_dim,
            inner_dim,
            num_heads,
            is_causal=True,
            dropout_p=dropout_p,
        )

        self.pre_cross_attn_norm = LayerNorm(embedding_dim)
        self.cross_attn = nn.MultiheadAttention(
            embedding_dim,
            num_heads,
            dropout=dropout_p,
            batch_first=True,
        )

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        padding_mask: torch.Tensor | None = None,
        memory_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # self-attention
        x = x + self.attn(self.pre_attn_norm(x), padding_mask)

        # cross-attention
        attended, _ = self.cross_attn(
            self.pre_cross_attn_norm(x),
            memory,
            memory,
            key_padding_mask=memory_padding_mask,
            need_weights=False,
        )
        x = x + attended

        # feed-forward network
        x = x + self.ffn(self.pre_ffn_norm(x))

        return x

## Positional Encoding

Осталась одна существенная проблема — наша модель пока ничего не знает о расположении токенов в тексте. Чтобы это исправить, введем позиционное кодирование (англ. *positional encoding*). В оригинальной архитектуре трансформера для этого использовались синусы и косинусы разных частот:

$$
\begin{aligned}
PE_{(pos,\,2i)} &= \sin\left(\frac{pos}{10000^{2i/d}}\right), \\
PE_{(pos,\,2i+1)} &= \cos\left(\frac{pos}{10000^{2i/d}}\right),
\end{aligned}
$$

где $pos$ — позиция токена, $d$ — размерность эмбеддинга, а $i$ — индекс пары компонент. Нумерация начинается с нуля: для четных компонент используется синус, для нечетных — косинус. Полученный позиционный вектор складывается с эмбеддингом токена перед подачей в первый слой трансформера.

Подробнее о различных вариантах позиционного кодирования можно прочитать в статье [You could have designed state of the art positional encoding](https://huggingface.co/blog/designing-positional-encoding).

In [ ]:
class PositionalEncoding(nn.Module):
    encoding: torch.Tensor

    def __init__(self, embedding_dim: int, max_length: int) -> None:
        super().__init__()

        positions = torch.arange(max_length).unsqueeze(1)
        dimensions = torch.arange(0, embedding_dim, 2)
        angles = positions / (10_000 ** (dimensions / embedding_dim))

        encoding = torch.zeros(max_length, embedding_dim)
        encoding[:, 0::2] = angles.sin()
        encoding[:, 1::2] = angles[:, : embedding_dim // 2].cos()

        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] > self.encoding.shape[1]:
            msg = "sequence length exceeds `max_length`."
            raise ValueError(msg)

        return x + self.encoding[:, : x.shape[1]].to(dtype=x.dtype)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

positional_encoding = PositionalEncoding(32, 128).eval()

plt.figure(figsize=(8, 4))
sns.heatmap(
    positional_encoding.encoding.squeeze(0),
    cmap="gray",
    vmin=-1,
    vmax=1,
    xticklabels=2,
    yticklabels=16,
    cbar_kws={"label": "PE value"},
)

plt.xlabel("Embedding dimension")
plt.ylabel("Token position")

plt.title("Sinusoidal positional encoding")

plt.tight_layout()
plt.show()

In [ ]:
from lightning import LightningModule


class Seq2SeqTransformer(LightningModule):
    def __init__(
        self,
        source_vocab_size: int,
        target_vocab_size: int,
        embedding_dim: int = 128,
        inner_dim: int = 512,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout_p: float = 0.1,
        source_pad_token_id: int = 0,
        target_pad_token_id: int = 0,
        max_length: int = 256,
    ) -> None:
        super().__init__()

        self.source_pad_token_id = source_pad_token_id
        self.target_pad_token_id = target_pad_token_id

        self.embedding_scale = embedding_dim**0.5

        self.source_embedding = nn.Embedding(source_vocab_size, embedding_dim, padding_idx=source_pad_token_id)
        self.target_embedding = nn.Embedding(target_vocab_size, embedding_dim, padding_idx=target_pad_token_id)

        self.positional_encoding = PositionalEncoding(embedding_dim, max_length)
        self.embedding_dropout = nn.Dropout(dropout_p)

        self.encoder = nn.ModuleList(
            [
                TransformerBlock(embedding_dim, inner_dim, num_heads, dropout_p=dropout_p)
                for _ in range(num_layers)
            ]
        )  # fmt: skip
        self.decoder = nn.ModuleList(
            [TransformerDecoderBlock(embedding_dim, inner_dim, num_heads, dropout_p) for _ in range(num_layers)]
        )  # fmt: sip

        self.encoder_norm = LayerNorm(embedding_dim)
        self.decoder_norm = LayerNorm(embedding_dim)

        self.head = nn.Linear(embedding_dim, target_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=target_pad_token_id)

    def encode(self, source_ids: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        padding_mask = source_ids == self.source_pad_token_id
        memory = self.positional_encoding(self.source_embedding(source_ids) * self.embedding_scale)
        memory = self.embedding_dropout(memory)
        for block in self.encoder:
            memory = block(memory, padding_mask)
        return self.encoder_norm(memory), padding_mask

    def decode(
        self,
        target_ids: torch.Tensor,
        memory: torch.Tensor,
        memory_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        padding_mask = target_ids == self.target_pad_token_id
        x = self.positional_encoding(self.target_embedding(target_ids) * self.embedding_scale)
        x = self.embedding_dropout(x)
        for block in self.decoder:
            x = block(x, memory, padding_mask, memory_padding_mask)
        return self.head(self.decoder_norm(x))

    def forward(self, source_ids: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
        memory, padding_mask = self.encode(source_ids)
        return self.decode(target_ids, memory, padding_mask)

    def training_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        source_ids, target_ids = batch
        logits = self(source_ids, target_ids[:, :-1])
        loss = self.loss_fn(logits.transpose(1, 2), target_ids[:, 1:])
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self) -> dict:
        optimizer = torch.optim.AdamW(self.parameters(), lr=3e-4)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=3e-4,
            total_steps=self.trainer.estimated_stepping_batches,
            pct_start=0.1,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    @torch.no_grad()
    def generate(
        self,
        source_ids: torch.Tensor,
        bos_token_id: int,
        max_new_tokens: int = 64,
    ) -> torch.Tensor:
        memory, padding_mask = self.encode(source_ids)
        generated = source_ids.new_full((source_ids.shape[0], 1), bos_token_id)
        for _ in range(max_new_tokens):
            logits = self.decode(generated, memory, padding_mask)
            next_ids = logits[:, -1].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_ids], dim=1)
        return generated

In [ ]:
from lightning import Trainer

model = Seq2SeqTransformer(
    ru_tokenizer.get_vocab_size(),
    en_tokenizer.get_vocab_size(),
    source_pad_token_id=ru_tokenizer.token_to_id("<pad>"),
    target_pad_token_id=en_tokenizer.token_to_id("<pad>"),
)


trainer = Trainer(max_epochs=20, gradient_clip_val=1.0)
trainer.fit(model, train_dataloaders=train_dataloader)

In [ ]:
import evaluate
from tqdm.auto import tqdm

bleu = evaluate.load("sacrebleu")

model.eval()

bos_token_id = en_tokenizer.token_to_id("<bos>")
eos_token_id = en_tokenizer.token_to_id("<eos>")

predictions, references = [], []

for source_ids, target_ids in tqdm(test_dataloader, desc="Test translations"):
    generated = model.generate(source_ids.to(model.device), bos_token_id=bos_token_id)

    for token_ids in generated[:, 1:].cpu().tolist():
        if eos_token_id in token_ids:
            token_ids = token_ids[: token_ids.index(eos_token_id) + 1]

        predictions.append(en_tokenizer.decode(token_ids))

    references.extend(en_tokenizer.decode_batch(target_ids.tolist()))

bleu_score = bleu.compute(predictions=predictions, references=[[text] for text in references])["score"]
print(f"Test BLEU: {bleu_score:.2f}")

In [ ]:
pd.DataFrame({"reference": references, "prediction": predictions}).head(5)

In [ ]:
example_idx = 221

source_ids, _ = test_dataset[example_idx]
source_ids = source_ids.unsqueeze(0).to(model.device)

token_ids = model.generate(source_ids, bos_token_id=bos_token_id).squeeze(0).tolist()
token_ids = token_ids[: token_ids.index(eos_token_id)]

decoder_ids = torch.tensor([[bos_token_id] + token_ids[:-1]], device=model.device)

model.eval()

with torch.no_grad():
    memory, source_padding_mask = model.encode(source_ids)
    padding_mask = decoder_ids == model.target_pad_token_id

    x = model.target_embedding(decoder_ids) * model.embedding_scale
    x = model.embedding_dropout(model.positional_encoding(x))

    for block in model.decoder[:-1]:
        x = block(x, memory, padding_mask, source_padding_mask)

    block = model.decoder[-1]
    x = x + block.attn(block.pre_attn_norm(x), padding_mask)
    _, attention_weights = block.cross_attn(
        block.pre_cross_attn_norm(x),
        memory,
        memory,
        key_padding_mask=source_padding_mask,
        need_weights=True,
        average_attn_weights=True,
    )

source_tokens = [ru_tokenizer.id_to_token(i) for i in source_ids[0].tolist()]
target_tokens = [en_tokenizer.id_to_token(i) for i in token_ids]

print("Source:", ru_tokenizer.decode(source_ids[0].tolist()))
print("Reference:", references[example_idx])
print("Prediction:", predictions[example_idx])

plt.figure(figsize=(6, 6))
sns.heatmap(
    attention_weights[0].cpu().numpy(),
    xticklabels=source_tokens,
    yticklabels=target_tokens,
    cmap="gray_r",
    vmin=0,
    # cbar_kws={"label": "Attention weight"},
    square=True,
)

plt.xlabel("Source tokens")
plt.ylabel("Generated tokens")

plt.title("Cross-attention")

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

## Last but not least

Мы разобрали архитектуру encoder-decoder трансформера для задачи Seq2Seq. Заметим, однако, что основная задача encoder — получить качественное представление исходного текста, а decoder — научиться моделировать целевой язык. Получается, мы можем использовать encoder или decoder как отдельную модель для решения других задач.

В действительности **encoder-only** архитектуры лежат в основе многих моделей для получения векторных представлений текстов (англ. *embedding models*). Один из самых известных примеров такой архитектуры — BERT.

В то же время **decoder-only** архитектуры стали основой современных больших языковых моделей (англ. *large language models*, LLM). GPT, Llama, DeepSeek и многие другие — это decoder-only модели.

Но подробнее об этом мы поговорим уже на следующих семинарах. Спасибо за внимание!
